In [1]:
from autogluon.tabular import TabularPredictor, TabularDataset
from sklearn.model_selection import train_test_split
from imblearn.combine import SMOTEENN
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
import pandas as pd

# Read only the header (first row)
df1 = pd.read_csv('busson20.csv', nrows=0)
df2 = pd.read_csv('Ryadh20.csv', nrows=0)

# Compare column names
if list(df1.columns) == list(df2.columns):
    print("Both CSV files have the same column names (order matters).")
else:
    print("CSV files have different column names.")

Both CSV files have the same column names (order matters).


In [3]:
data = TabularDataset("busson20.csv")

data.columns = data.columns.str.replace(r"[ ()]", "", regex=True)
data = data[data["ConnectedDevicesMAC"] != 0]

label = "DeviceScreenState"
data = TabularDataset(data)

In [4]:
predictor = TabularPredictor(label=label, eval_metric="accuracy").fit(data, presets="good")

No path specified. Models will be saved in: "AutogluonModels\ag-20251210_131009"
Preset alias specified: 'good' maps to 'good_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.6
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Memory Avail:       1.34 GB / 7.62 GB (17.7%)
Disk Space Avail:   18.64 GB / 163.04 GB (11.4%)
Presets specified: ['good']
Using hyperparameters preset: hyperparameters='light'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid

(autoscaler +9m1s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.
(autoscaler +9m1s) Warning: The following resource request cannot be scheduled right now: {'CPU': 8.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.


(_dystack pid=17248) 	0.9977	 = Validation score   (accuracy)
(_dystack pid=17248) 	39.33s	 = Training   runtime
(_dystack pid=17248) 	0.03s	 = Validation runtime
(_dystack pid=17248) Fitting model: ExtraTreesGini_BAG_L2 ... Training model for up to 341.27s of the 341.25s of remaining time.
(_ray_fit pid=17976) Warning: Low available memory may cause OOM error if training continues [repeated 7x across cluster]
(_ray_fit pid=17976) Available Memory: 496 MB [repeated 7x across cluster]
(_ray_fit pid=17976) Estimated GBM model size: 0 MB [repeated 7x across cluster]
(_ray_fit pid=17976) Warning: Early stopped GBM model prior to optimal result to avoid OOM error. Please increase available memory to avoid subpar model quality. [repeated 7x across cluster]
(_dystack pid=17248) 	0.9976	 = Validation score   (accuracy)
(_dystack pid=17248) 	1.21s	 = Training   runtime
(_dystack pid=17248) 	0.41s	 = Validation runtime
(_dystack pid=17248) Fitting model: ExtraTreesEntr_BAG_L2 ... Training model 

KeyboardInterrupt: 

(_dystack pid=17248) 	0.9978	 = Validation score   (accuracy)
(_dystack pid=17248) 	115.14s	 = Training   runtime
(_dystack pid=17248) 	0.81s	 = Validation runtime
(_dystack pid=17248) Fitting model: LightGBMLarge_BAG_L2 ... Training model for up to 157.38s of the 157.36s of remaining time.
(_dystack pid=17248) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=7.59%)
(_ray_fit pid=14792) Warning: Low available memory may cause OOM error if training continues
(_ray_fit pid=14792) Available Memory: 297 MB
(_ray_fit pid=14792) Estimated GBM model size: 0 MB
(_ray_fit pid=14792) Warning: Early stopped GBM model prior to optimal result to avoid OOM error. Please increase available memory to avoid subpar model quality.
(_dystack pid=17248) 	0.6414	 = Validation score   (accuracy)
(_dystack pid=17248) 	1.56s	 = Training   runtime
(_dystack pid=17248) 	0.03s	 = Validation runtime
(_dystack pid=17248) Fitting model: Wei

(raylet) The node with node id: 9ebf76612d02044c793b451b1e11f595b73b0681d9bdfc4a12e64942 and address: 127.0.0.1 and node name: 127.0.0.1 has been marked dead because the detector has missed too many heartbeats from it. This can happen when a 	(1) raylet crashes unexpectedly (OOM, etc.) 
	(2) raylet has lagging heartbeats due to slow network or busy workload.


(raylet) [2025-12-11 20:52:09,211 E 13836 13708] (raylet.exe) agent_manager.cc:84: The raylet exited immediately because one Ray agent failed, agent_name = dashboard_agent/15724.
(raylet) The raylet fate shares with the agent. This can happen because
(raylet) - The version of `grpcio` doesn't follow Ray's requirement. Agent can segfault with the incorrect `grpcio` version. Check the grpcio version `pip freeze | grep grpcio`.
(raylet) - The agent failed to start because of unexpected error or port conflict. Read the log `cat /tmp/ray/session_latest/logs/{dashboard_agent|runtime_env_agent}.log`. You can find the log file structure here https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#logging-directory-structure.
(raylet) - The agent is killed by the OS (e.g., out of memory).
(raylet) [2025-12-11 20:52:09,530 C 13836 15748] (raylet.exe) node_manager.cc:575: GCS consider this node to be dead. This may happen when GCS is not backed by a DB and restarted or 

In [ ]:
validation = TabularDataset("busson20 copy.csv")
validation.columns = data.columns.str.replace(r"[ ()]", "", regex=True)

invalid_rows = validation[validation["ConnectedDevicesMAC"] == 0].copy()
valid_rows = validation[validation["ConnectedDevicesMAC"] != 0].copy()

In [ ]:
leaderboard = predictor.leaderboard(
    valid_rows,
    extra_metrics=["...accuracy", "recall", "precision", "f1", "roc_auc"],
    silent=True
)

In [ ]:
max_score_test = leaderboard["score_test"].max() and filter top_models

In [ ]:
best_model_results = predictor.evaluate(valid_rows, model=best_model, silent=True)

In [ ]:
importance_df = predictor.feature_importance(data, model=best_model)

In [ ]:
features_to_remove = importance_df[importance_df["importance"] <= 0].index.tolist()
data_reduced = data.drop(columns=features_to_remove)

In [ ]:
predictor_reduced = TabularPredictor(label=label, eval_metric="accuracy").fit(data_reduced, presets="good")